In [1]:
def preprocess_image(path, debug=False):
    """
    Converts a phone photo of a handwritten digit into a
    784-length vector matching X's format above.
    """
    img = Image.open(path).convert("L")           # to grayscale
    img = ImageOps.invert(img)                     # photo is ink-on-paper -> MNIST is white-on-black

    img_np = np.array(img)
    threshold = 50
    img_np = np.where(img_np > threshold, img_np, 0)   # clean up shadows/noise

    # Crop tightly to the digit
    coords = np.column_stack(np.where(img_np > threshold))
    if coords.size > 0:
        y0, x0 = coords.min(axis=0)
        y1, x1 = coords.max(axis=0)
        img_np = img_np[y0:y1+1, x0:x1+1]

    # Pad to square, then add margin (MNIST digits have some border, not edge-to-edge)
    h, w = img_np.shape
    size = max(h, w)
    padded = np.zeros((size, size), dtype=np.uint8)
    y_off, x_off = (size - h) // 2, (size - w) // 2
    padded[y_off:y_off+h, x_off:x_off+w] = img_np

    border = size // 5
    bordered = np.zeros((size + 2*border, size + 2*border), dtype=np.uint8)
    bordered[border:border+size, border:border+size] = padded

    final_img = Image.fromarray(bordered).resize((28, 28), Image.LANCZOS)
    final_np = np.array(final_img).astype("float64")

    if debug:
        plt.imshow(final_np, cmap=mpl.cm.binary)
        plt.title("Preprocessed input")
        plt.show()

    return final_np.reshape(1, -1)   # shape (1, 784), same as some_digit


def predict_digit(path):
    clf = joblib.load("mnist_voting_clf.pkl")
    x = preprocess_image(path, debug=True)
    pred = clf.predict(x)[0]
    probs = clf.predict_proba(x)[0]
    print(f"Predicted digit: {pred}  (confidence: {probs[pred]:.2%})")
    return pred, probs

# Example:
# predict_digit("my_digit_photo.jpg")